In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../data/adn_boca_real_features.csv')

print(f"Dataset: {df.shape}")
print(df.isnull().sum())

In [ ]:
df.info()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# rating y rendimiento NO se usan porque etiqueta se define desde rating (data leakage)
FEATURES_CLUSTER = ["goles_por_partido", "pases_norm", "participacion_gol"]
FEATURES_MODEL   = FEATURES_CLUSTER + ["asist_por_partido",
                                         "perfil_ofensivo", "edad", "partidos", "experiencia"]

# ── Paso 1: Clustering sin etiqueta ──
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURES_CLUSTER])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

# Mapeo manual al revisar centroides (ajustar según resultados)
print(df.groupby("cluster")[FEATURES_CLUSTER + ["etiqueta"]].mean().round(2))

# ── Paso 2: Clasificador con cluster como feature ──
X = df[FEATURES_MODEL + ["cluster"]]
y = df["etiqueta"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = XGBClassifier(
    n_estimators=100,
    max_depth=4,          # limitado para evitar overfitting
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred,
      target_names=["No encaja", "Perfil Boca"]))

In [ ]:
# Pruebas de hiperparametros
# rating y rendimiento NO se usan porque etiqueta se define desde rating (data leakage)
FEATURES_CLUSTER = ["goles_por_partido", "pases_norm", "participacion_gol"]
FEATURES_MODEL   = FEATURES_CLUSTER + [
    "asist_por_partido",
    "perfil_ofensivo", "edad", "partidos", "experiencia"
]

# ── Paso 1: Clustering sin etiqueta ──
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURES_CLUSTER])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

print(df.groupby("cluster")[FEATURES_CLUSTER + ["etiqueta"]].mean().round(2))

# ── Paso 2: Clasificador ──
X = df[FEATURES_MODEL + ["cluster"]]
y = df["etiqueta"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="logloss",
    random_state=42
)

# activar early stopping para versiones viejas
model.set_params(early_stopping_rounds=20)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("Best iteration:", model.best_iteration)
y_pred = model.predict(X_test)
print(classification_report(
    y_test, y_pred,
    target_names=["No encaja", "Perfil Boca"]
))



In [ ]:
y_train_pred = model.predict(X_train)
print(classification_report(y_train, y_train_pred))


In [ ]:
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression(max_iter=500)
logreg.fit(X_train, y_train)
print(classification_report(y_test, logreg.predict(X_test)))
